# 04 — Diagnóstico de Negócio: Queda de Vendas

## Objetivo

Consolidar e validar as evidências obtidas nas etapas de análise exploratória
e análise estatística, verificando a consistência das métricas, a
comparabilidade dos períodos e os limites das conclusões.

## Pergunta de negócio

Quais padrões observados nos dados ajudam a explicar a deterioração
do faturamento entre maio e junho de 2018?

## Princípios da análise

- Validar os indicadores antes de interpretá-los.
- Diferenciar evidências descritivas de evidências estatísticas.
- Não confundir associação com causalidade.
- Registrar definições, denominadores e limitações.
- Não impor aos dados uma explicação previamente escolhida.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda valor: f"{valor:,.4f}")

print("Ambiente de análise preparado.")
print(f"Versão do pandas: {pd.__version__}")
print(f"Versão do NumPy: {np.__version__}")

Ambiente de análise preparado.
Versão do pandas: 3.0.6
Versão do NumPy: 2.5.3


In [2]:
from pathlib import Path

# Identifica a pasta a partir da qual o notebook está sendo executado
pasta_atual = Path.cwd()

print("Pasta de execução:")
print(pasta_atual)

print("\nConteúdo da pasta atual:")
for item in sorted(pasta_atual.iterdir()):
    tipo = "Pasta" if item.is_dir() else "Arquivo"
    print(f"{tipo}: {item.name}")

Pasta de execução:
c:\Users\joana\OneDrive\Área de Trabalho\case1-queda-vendas\notebooks

Conteúdo da pasta atual:
Arquivo: 01_data_profiling.ipynb
Arquivo: 02_eda.ipynb
Arquivo: 03_statistical_analysis.ipynb
Arquivo: 04_diagnosis.ipynb


In [3]:
# A pasta atual é "notebooks".
# Portanto, o diretório pai deve ser a raiz do projeto.

raiz_projeto = Path.cwd().parent
pasta_dados_brutos = raiz_projeto / "data" / "raw"

print("Raiz do projeto:")
print(raiz_projeto)

print("\nA raiz do projeto existe?")
print(raiz_projeto.exists())

print("\nPasta de dados brutos:")
print(pasta_dados_brutos)

print("\nA pasta de dados brutos existe?")
print(pasta_dados_brutos.exists())

if pasta_dados_brutos.exists():
    print("\nArquivos encontrados em data/raw:")
    for arquivo in sorted(pasta_dados_brutos.iterdir()):
        if arquivo.is_file():
            print(f"- {arquivo.name}")

Raiz do projeto:
c:\Users\joana\OneDrive\Área de Trabalho\case1-queda-vendas

A raiz do projeto existe?
True

Pasta de dados brutos:
c:\Users\joana\OneDrive\Área de Trabalho\case1-queda-vendas\data\raw

A pasta de dados brutos existe?
True

Arquivos encontrados em data/raw:
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


In [4]:
# Caminhos dos arquivos necessários para a validação inicial
arquivo_pedidos = (
    pasta_dados_brutos / "olist_orders_dataset.csv"
)

arquivo_itens = (
    pasta_dados_brutos / "olist_order_items_dataset.csv"
)

# Carregamento das tabelas originais
pedidos = pd.read_csv(
    arquivo_pedidos,
    parse_dates=["order_purchase_timestamp"]
)

itens = pd.read_csv(arquivo_itens)

# Verificação inicial das tabelas
print("TABELA DE PEDIDOS")
print(f"Dimensões: {pedidos.shape}")
print(f"Colunas: {pedidos.columns.tolist()}")

print("\nTABELA DE ITENS")
print(f"Dimensões: {itens.shape}")
print(f"Colunas: {itens.columns.tolist()}")

TABELA DE PEDIDOS
Dimensões: (99441, 8)
Colunas: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

TABELA DE ITENS
Dimensões: (112650, 7)
Colunas: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


In [5]:
# 1. Integridade da tabela de pedidos
print("=== TABELA DE PEDIDOS ===")

print(
    "Pedidos registrados:",
    len(pedidos)
)

print(
    "IDs de pedidos distintos:",
    pedidos["order_id"].nunique()
)

print(
    "IDs de pedidos duplicados:",
    pedidos["order_id"].duplicated().sum()
)

print(
    "IDs de pedidos ausentes:",
    pedidos["order_id"].isna().sum()
)

print(
    "Datas de compra ausentes:",
    pedidos["order_purchase_timestamp"].isna().sum()
)


# 2. Integridade da tabela de itens
print("\n=== TABELA DE ITENS ===")

print(
    "Linhas de itens:",
    len(itens)
)

print(
    "IDs de pedidos distintos nos itens:",
    itens["order_id"].nunique()
)

print(
    "IDs de pedidos ausentes nos itens:",
    itens["order_id"].isna().sum()
)

print(
    "IDs de produtos ausentes:",
    itens["product_id"].isna().sum()
)

print(
    "Preços ausentes:",
    itens["price"].isna().sum()
)

print(
    "Preços iguais ou inferiores a zero:",
    itens["price"].le(0).sum()
)


# 3. Verificação da chave composta dos itens
print("\n=== CHAVE DOS ITENS ===")

print(
    "Pares order_id + order_item_id duplicados:",
    itens.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

=== TABELA DE PEDIDOS ===
Pedidos registrados: 99441
IDs de pedidos distintos: 99441
IDs de pedidos duplicados: 0
IDs de pedidos ausentes: 0
Datas de compra ausentes: 0

=== TABELA DE ITENS ===
Linhas de itens: 112650
IDs de pedidos distintos nos itens: 98666
IDs de pedidos ausentes nos itens: 0
IDs de produtos ausentes: 0
Preços ausentes: 0
Preços iguais ou inferiores a zero: 0

=== CHAVE DOS ITENS ===
Pares order_id + order_item_id duplicados: 0


In [6]:
# Identificadores dos pedidos em cada tabela
ids_pedidos = set(pedidos["order_id"])
ids_pedidos_com_itens = set(itens["order_id"])

# Pedidos registrados sem correspondência na tabela de itens
ids_sem_itens = ids_pedidos - ids_pedidos_com_itens

print("=== CONCILIAÇÃO ENTRE PEDIDOS E ITENS ===")

print(f"Pedidos na tabela de pedidos: {len(ids_pedidos):,}")
print(f"Pedidos com itens registrados: {len(ids_pedidos_com_itens):,}")
print(f"Pedidos sem itens registrados: {len(ids_sem_itens):,}")

# Examina os status desses pedidos
pedidos_sem_itens = pedidos[
    pedidos["order_id"].isin(ids_sem_itens)
]

print("\nStatus dos pedidos sem itens:")
print(
    pedidos_sem_itens["order_status"]
    .value_counts(dropna=False)
)

print("\nPercentual dos pedidos sem itens:")
print(f"{len(ids_sem_itens) / len(pedidos) * 100:.2f}%")

=== CONCILIAÇÃO ENTRE PEDIDOS E ITENS ===
Pedidos na tabela de pedidos: 99,441
Pedidos com itens registrados: 98,666
Pedidos sem itens registrados: 775

Status dos pedidos sem itens:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

Percentual dos pedidos sem itens:
0.78%


In [7]:
# 1. Identificar o mês de compra de cada pedido
pedidos["mes_compra"] = (
    pedidos["order_purchase_timestamp"]
    .dt.to_period("M")
)

# 2. Calcular a receita mensal dos itens
receita_mensal = (
    itens
    .merge(
        pedidos[["order_id", "mes_compra"]],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("mes_compra")["price"]
    .sum()
    .rename("receita_itens")
)

# 3. Contar pedidos distintos por mês
pedidos_mensais = (
    pedidos
    .groupby("mes_compra")["order_id"]
    .nunique()
    .rename("pedidos_registrados")
)

# 4. Contar pedidos distintos que possuem itens registrados
pedidos_com_itens_mensais = (
    itens
    .merge(
        pedidos[["order_id", "mes_compra"]],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .groupby("mes_compra")["order_id"]
    .nunique()
    .rename("pedidos_com_itens")
)

print("Bases mensais preparadas.")
print("Meses encontrados:", len(pedidos_mensais))
print("Período inicial:", pedidos_mensais.index.min())
print("Período final:", pedidos_mensais.index.max())

Bases mensais preparadas.
Meses encontrados: 25
Período inicial: 2016-09
Período final: 2018-10


In [8]:
# Selecionar os dois meses da análise
meses_analise = ["2018-05", "2018-06"]

# Montar a tabela comparativa dos indicadores
comparativo = pd.concat(
    [
        receita_mensal,
        pedidos_mensais,
        pedidos_com_itens_mensais
    ],
    axis=1
)

# Selecionar maio e junho de 2018
comparativo = comparativo.loc[
    meses_analise
].copy()

# Calcular o ticket operacional por pedido com itens
comparativo["ticket_por_pedido_com_itens"] = (
    comparativo["receita_itens"]
    / comparativo["pedidos_com_itens"]
)

# Calcular as variações percentuais de maio para junho
variacao_receita = (
    comparativo.loc["2018-06", "receita_itens"]
    / comparativo.loc["2018-05", "receita_itens"]
    - 1
) * 100

variacao_pedidos = (
    comparativo.loc["2018-06", "pedidos_registrados"]
    / comparativo.loc["2018-05", "pedidos_registrados"]
    - 1
) * 100

variacao_pedidos_com_itens = (
    comparativo.loc["2018-06", "pedidos_com_itens"]
    / comparativo.loc["2018-05", "pedidos_com_itens"]
    - 1
) * 100

variacao_ticket = (
    comparativo.loc["2018-06", "ticket_por_pedido_com_itens"]
    / comparativo.loc["2018-05", "ticket_por_pedido_com_itens"]
    - 1
) * 100

print("=== COMPARATIVO: MAIO × JUNHO DE 2018 ===")
display(comparativo.round(2))

print("\n=== VARIAÇÕES PERCENTUAIS ===")
print(f"Receita dos itens: {variacao_receita:.2f}%")
print(f"Pedidos registrados: {variacao_pedidos:.2f}%")
print(f"Pedidos com itens: {variacao_pedidos_com_itens:.2f}%")
print(f"Ticket por pedido com itens: {variacao_ticket:.2f}%")

=== COMPARATIVO: MAIO × JUNHO DE 2018 ===


,receita_itens,pedidos_registrados,pedidos_com_itens,ticket_por_pedido_com_itens
mes_compra,,,,
2018-05,"996,517.6800",6873,"6,853.0000",145.4100
2018-06,"865,124.3100",6167,"6,160.0000",140.4400



=== VARIAÇÕES PERCENTUAIS ===
Receita dos itens: -13.19%
Pedidos registrados: -10.27%
Pedidos com itens: -10.11%
Ticket por pedido com itens: -3.42%


In [9]:
# Verificar a identidade matemática da receita
comparativo["receita_recalculada"] = (
    comparativo["pedidos_com_itens"]
    * comparativo["ticket_por_pedido_com_itens"]
)

comparativo["diferenca_receita"] = (
    comparativo["receita_itens"]
    - comparativo["receita_recalculada"]
)

print("=== VALIDAÇÃO DA IDENTIDADE DA RECEITA ===")

display(
    comparativo[
        [
            "receita_itens",
            "pedidos_com_itens",
            "ticket_por_pedido_com_itens",
            "receita_recalculada",
            "diferenca_receita"
        ]
    ].round(4)
)

print(
    "\nMaior diferença absoluta:",
    comparativo["diferenca_receita"].abs().max()
)

=== VALIDAÇÃO DA IDENTIDADE DA RECEITA ===


,receita_itens,pedidos_com_itens,ticket_por_pedido_com_itens,receita_recalculada,diferenca_receita
mes_compra,,,,,
2018-05,"996,517.6800","6,853.0000",145.4133,"996,517.6800",0.0000
2018-06,"865,124.3100","6,160.0000",140.4423,"865,124.3100",0.0000



Maior diferença absoluta: 0.0


In [10]:
# Valores de maio e junho
receita_maio = comparativo.loc["2018-05", "receita_itens"]
receita_junho = comparativo.loc["2018-06", "receita_itens"]

pedidos_maio = comparativo.loc["2018-05", "pedidos_com_itens"]
pedidos_junho = comparativo.loc["2018-06", "pedidos_com_itens"]

ticket_maio = comparativo.loc[
    "2018-05", "ticket_por_pedido_com_itens"
]
ticket_junho = comparativo.loc[
    "2018-06", "ticket_por_pedido_com_itens"
]

# Diferenças absolutas
diferenca_receita = receita_junho - receita_maio
diferenca_pedidos = pedidos_junho - pedidos_maio
diferenca_ticket = ticket_junho - ticket_maio

# Variações percentuais
variacao_receita = diferenca_receita / receita_maio * 100
variacao_pedidos = diferenca_pedidos / pedidos_maio * 100
variacao_ticket = diferenca_ticket / ticket_maio * 100

print("=== IMPACTO DA VARIAÇÃO: MAIO → JUNHO DE 2018 ===")

print(f"Diferença de receita: R$ {diferenca_receita:,.2f}")
print(f"Variação de receita: {variacao_receita:.2f}%")

print(f"\nDiferença de pedidos com itens: {diferenca_pedidos:,}")
print(f"Variação de pedidos com itens: {variacao_pedidos:.2f}%")

print(f"\nDiferença do ticket por pedido com itens: R$ {diferenca_ticket:,.2f}")
print(f"Variação do ticket: {variacao_ticket:.2f}%")

=== IMPACTO DA VARIAÇÃO: MAIO → JUNHO DE 2018 ===
Diferença de receita: R$ -131,393.37
Variação de receita: -13.19%

Diferença de pedidos com itens: -693.0
Variação de pedidos com itens: -10.11%

Diferença do ticket por pedido com itens: R$ -4.97
Variação do ticket: -3.42%


In [11]:
# Limites temporais dos meses analisados
for mes in ["2018-05", "2018-06"]:
    inicio_mes = pd.Period(mes, freq="M").start_time
    fim_mes = pd.Period(mes, freq="M").end_time

    pedidos_mes = pedidos[
        pedidos["mes_compra"] == pd.Period(mes, freq="M")
    ]

    datas_observadas = pedidos_mes[
        "order_purchase_timestamp"
    ]

    print(f"=== {mes} ===")
    print(f"Início do mês: {inicio_mes}")
    print(f"Fim do mês: {fim_mes}")
    print(f"Primeira compra observada: {datas_observadas.min()}")
    print(f"Última compra observada: {datas_observadas.max()}")
    print(f"Pedidos registrados: {len(pedidos_mes):,}")
    print()

=== 2018-05 ===
Início do mês: 2018-05-01 00:00:00
Fim do mês: 2018-05-31 23:59:59.999999
Primeira compra observada: 2018-05-01 00:02:11
Última compra observada: 2018-05-31 23:51:24
Pedidos registrados: 6,873

=== 2018-06 ===
Início do mês: 2018-06-01 00:00:00
Fim do mês: 2018-06-30 23:59:59.999999
Primeira compra observada: 2018-06-01 00:39:55
Última compra observada: 2018-06-30 23:59:49
Pedidos registrados: 6,167



In [12]:

# Consolidar os principais indicadores de maio e junho de 2018

maio = comparativo.loc["2018-05"]
junho = comparativo.loc["2018-06"]

indicadores = {
    "Receita de itens (R$)": "receita_itens",
    "Pedidos registrados": "pedidos_registrados",
    "Pedidos com itens": "pedidos_com_itens",
    "Receita por pedido com itens (R$)": "ticket_por_pedido_com_itens",
}

registros = []

for nome, coluna in indicadores.items():
    valor_maio = maio[coluna]
    valor_junho = junho[coluna]

    registros.append({
        "indicador": nome,
        "maio_2018": valor_maio,
        "junho_2018": valor_junho,
        "variacao_absoluta": valor_junho - valor_maio,
        "variacao_percentual": (
            (valor_junho / valor_maio) - 1
        ) * 100
    })

matriz_evidencias = pd.DataFrame(registros)

display(matriz_evidencias.round(2))


,indicador,maio_2018,junho_2018,variacao_absoluta,variacao_percentual
0,Receita de itens (R$),"996,517.6800","865,124.3100","-131,393.3700",-13.1900
1,Pedidos registrados,"6,873.0000","6,167.0000",-706.0000,-10.2700
2,Pedidos com itens,"6,853.0000","6,160.0000",-693.0000,-10.1100
3,Receita por pedido com itens (R$),145.4100,140.4400,-4.9700,-3.4200


In [13]:

# Validar a consistência matemática da receita

validacao_receita = comparativo[
    [
        "receita_itens",
        "pedidos_com_itens",
        "ticket_por_pedido_com_itens"
    ]
].copy()

validacao_receita["receita_recalculada"] = (
    validacao_receita["pedidos_com_itens"]
    * validacao_receita["ticket_por_pedido_com_itens"]
)

validacao_receita["diferenca"] = (
    validacao_receita["receita_itens"]
    - validacao_receita["receita_recalculada"]
)

display(validacao_receita.round(4))

print(
    "Maior diferença absoluta:",
    validacao_receita["diferenca"].abs().max()
)


,receita_itens,pedidos_com_itens,ticket_por_pedido_com_itens,receita_recalculada,diferenca
mes_compra,,,,,
2018-05,"996,517.6800","6,853.0000",145.4133,"996,517.6800",0.0000
2018-06,"865,124.3100","6,160.0000",140.4423,"865,124.3100",0.0000


Maior diferença absoluta: 0.0


In [14]:

# Identificar os DataFrames disponíveis para a próxima análise

dataframes_disponiveis = {
    nome: objeto.shape
    for nome, objeto in globals().copy().items()
    if isinstance(objeto, pd.DataFrame)
}

for nome, dimensoes in sorted(dataframes_disponiveis.items()):
    print(f"{nome}: {dimensoes}")


comparativo: (2, 6)
itens: (112650, 7)
matriz_evidencias: (4, 5)
pedidos: (99441, 9)
pedidos_mes: (6167, 9)
pedidos_sem_itens: (775, 8)
validacao_receita: (2, 5)


In [15]:

# Carregar as tabelas necessárias para analisar as categorias

arquivo_produtos = pasta_dados_brutos / "olist_products_dataset.csv"
arquivo_traducao = pasta_dados_brutos / "product_category_name_translation.csv"

produtos = pd.read_csv(arquivo_produtos)
traducao_categorias = pd.read_csv(arquivo_traducao)

print("=== Tabela de produtos ===")
print("Dimensões:", produtos.shape)
print("Colunas:", produtos.columns.tolist())
print("IDs de produtos únicos:", produtos["product_id"].nunique())
print("IDs de produtos ausentes:", produtos["product_id"].isna().sum())
print("IDs de produtos duplicados:", produtos["product_id"].duplicated().sum())
print("Categorias ausentes:", produtos["product_category_name"].isna().sum())

print("\n=== Tradução das categorias ===")
print("Dimensões:", traducao_categorias.shape)
print("Colunas:", traducao_categorias.columns.tolist())

display(produtos.head())
display(traducao_categorias.head())


=== Tabela de produtos ===
Dimensões: (32951, 9)
Colunas: ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
IDs de produtos únicos: 32951
IDs de produtos ausentes: 0
IDs de produtos duplicados: 0
Categorias ausentes: 610

=== Tradução das categorias ===
Dimensões: (71, 2)
Colunas: ['product_category_name', 'product_category_name_english']


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0000,287.0000,1.0000,225.0000,16.0000,10.0000,14.0000
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0000,276.0000,1.0000,"1,000.0000",30.0000,18.0000,20.0000
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0000,250.0000,1.0000,154.0000,18.0000,9.0000,15.0000
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0000,261.0000,1.0000,371.0000,26.0000,4.0000,26.0000
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0000,402.0000,4.0000,625.0000,20.0000,17.0000,13.0000


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [16]:

# Preparar os itens vendidos com mês da compra e categoria do produto

itens_com_categoria = (
    itens
    .merge(
        pedidos[["order_id", "mes_compra"]],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        produtos[["product_id", "product_category_name"]],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)

# Preservar vendas de produtos sem categoria informada
itens_com_categoria["categoria"] = (
    itens_com_categoria["product_category_name"]
    .fillna("categoria_nao_informada")
)

print("Dimensões dos itens originais:", itens.shape)
print("Dimensões após os cruzamentos:", itens_com_categoria.shape)

print(
    "Itens sem mês associado:",
    itens_com_categoria["mes_compra"].isna().sum()
)

print(
    "Itens sem categoria informada:",
    (
        itens_com_categoria["categoria"]
        == "categoria_nao_informada"
    ).sum()
)

print(
    "Receita original dos itens (R$):",
    round(itens["price"].sum(), 2)
)

print(
    "Receita após os cruzamentos (R$):",
    round(itens_com_categoria["price"].sum(), 2)
)

display(
    itens_com_categoria[
        ["order_id", "product_id", "mes_compra", "price", "categoria"]
    ].head()
)


Dimensões dos itens originais: (112650, 7)
Dimensões após os cruzamentos: (112650, 10)
Itens sem mês associado: 0
Itens sem categoria informada: 1603
Receita original dos itens (R$): 13591643.7
Receita após os cruzamentos (R$): 13591643.7


,order_id,product_id,mes_compra,price,categoria
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,2017-09,58.9000,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,2017-04,239.9000,pet_shop
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,2018-01,199.0000,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,2018-08,12.9900,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,2017-02,199.9000,ferramentas_jardim


In [17]:

# Selecionar os dois meses da análise
meses_analise = ["2018-05", "2018-06"]

itens_periodo = itens_com_categoria[
    itens_com_categoria["mes_compra"].astype(str).isin(meses_analise)
].copy()

# Calcular a receita por categoria em cada mês
receita_categoria = (
    itens_periodo
    .pivot_table(
        index="categoria",
        columns=itens_periodo["mes_compra"].astype(str),
        values="price",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(columns=meses_analise, fill_value=0)
    .rename(columns={
        "2018-05": "receita_maio",
        "2018-06": "receita_junho"
    })
)

# Calcular as variações
receita_categoria["variacao_absoluta"] = (
    receita_categoria["receita_junho"]
    - receita_categoria["receita_maio"]
)

receita_categoria["variacao_percentual"] = np.where(
    receita_categoria["receita_maio"] != 0,
    (
        receita_categoria["variacao_absoluta"]
        / receita_categoria["receita_maio"]
    ) * 100,
    np.nan
)

# Ordenar pelas maiores perdas em reais
receita_categoria = receita_categoria.sort_values(
    "variacao_absoluta"
)

print("Quantidade de categorias:", len(receita_categoria))

print("\n=== 10 categorias com maiores perdas ===")
display(receita_categoria.head(10).round(2))

print("\n=== 10 categorias com maiores crescimentos ===")
display(receita_categoria.tail(10).sort_values(
    "variacao_absoluta", ascending=False
).round(2))

print("\n=== Reconciliação da variação total ===")
print(
    "Variação total por categoria (R$):",
    round(receita_categoria["variacao_absoluta"].sum(), 2)
)


Quantidade de categorias: 67

=== 10 categorias com maiores perdas ===


mes_compra,receita_maio,receita_junho,variacao_absoluta,variacao_percentual
categoria,,,,
relogios_presentes,"123,872.6600","86,885.5600","-36,987.1000",-29.8600
ferramentas_jardim,"37,863.1800","18,018.7100","-19,844.4700",-52.4100
esporte_lazer,"59,664.1100","45,851.0100","-13,813.1000",-23.1500
moveis_decoracao,"54,260.9800","42,190.2200","-12,070.7600",-22.2500
cool_stuff,"30,039.1100","19,316.3700","-10,722.7400",-35.7000
brinquedos,"31,767.3000","21,221.4500","-10,545.8500",-33.2000
informatica_acessorios,"51,515.7300","42,326.4500","-9,189.2800",-17.8400
moveis_escritorio,"21,005.8600","12,725.5800","-8,280.2800",-39.4200
eletronicos,"16,989.2900","9,664.5700","-7,324.7200",-43.1100



=== 10 categorias com maiores crescimentos ===


mes_compra,receita_maio,receita_junho,variacao_absoluta,variacao_percentual
categoria,,,,
beleza_saude,"96,460.3600","107,908.8200","11,448.4600",11.8700
pet_shop,"11,708.4600","19,491.7800","7,783.3200",66.4800
eletroportateis,"10,620.9900","16,022.7000","5,401.7100",50.8600
sinalizacao_e_seguranca,683.5000,"5,863.1800","5,179.6800",757.8200
automotivo,"40,928.1400","45,407.4900","4,479.3500",10.9400
pcs,"9,429.9200","13,127.0000","3,697.0800",39.2100
perfumaria,"24,757.7000","27,335.8600","2,578.1600",10.4100
construcao_ferramentas_construcao,"20,748.6000","22,947.7200","2,199.1200",10.6000
climatizacao,"2,051.2000","3,567.9000","1,516.7000",73.9400



=== Reconciliação da variação total ===
Variação total por categoria (R$): -131393.37


In [18]:

# Validar a reconciliação da receita por categoria

variacao_total_categorias = receita_categoria["variacao_absoluta"].sum()

receita_maio_total = itens_periodo.loc[
    itens_periodo["mes_compra"].astype(str) == "2018-05",
    "price"
].sum()

receita_junho_total = itens_periodo.loc[
    itens_periodo["mes_compra"].astype(str) == "2018-06",
    "price"
].sum()

variacao_total_operacao = receita_junho_total - receita_maio_total

print(f"Variação somada das categorias: R$ {variacao_total_categorias:,.2f}")
print(f"Variação total da operação:     R$ {variacao_total_operacao:,.2f}")
print(
    "Diferença entre os resultados:  "
    f"R$ {variacao_total_categorias - variacao_total_operacao:,.2f}"
)

# Medir o peso das dez maiores perdas
dez_maiores_perdas = receita_categoria.head(10)["variacao_absoluta"].sum()

print(f"\nPerdas somadas das 10 categorias: R$ {dez_maiores_perdas:,.2f}")
print(
    "Percentual dessas perdas sobre a queda líquida: "
    f"{abs(dez_maiores_perdas / variacao_total_operacao) * 100:.2f}%"
)


Variação somada das categorias: R$ -131,393.37
Variação total da operação:     R$ -131,393.37
Diferença entre os resultados:  R$ -0.00

Perdas somadas das 10 categorias: R$ -135,440.90
Percentual dessas perdas sobre a queda líquida: 103.08%


In [19]:

# Selecionar as dez categorias com maiores perdas
categorias_prioritarias = receita_categoria.head(10).index

itens_prioritarios = itens_com_categoria[
    itens_com_categoria["categoria"].isin(categorias_prioritarias)
    & itens_com_categoria["mes_compra"].astype(str).isin(
        ["2018-05", "2018-06"]
    )
].copy()

# Calcular a receita por produto em cada mês
receita_produto = (
    itens_prioritarios
    .pivot_table(
        index=["product_id", "categoria"],
        columns=itens_prioritarios["mes_compra"].astype(str),
        values="price",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(columns=["2018-05", "2018-06"], fill_value=0)
    .rename(columns={
        "2018-05": "receita_maio",
        "2018-06": "receita_junho"
    })
    .reset_index()
)

# Calcular a variação absoluta por produto
receita_produto["variacao_absoluta"] = (
    receita_produto["receita_junho"]
    - receita_produto["receita_maio"]
)

# Calcular a variação percentual
receita_produto["variacao_percentual"] = np.where(
    receita_produto["receita_maio"] > 0,
    (
        receita_produto["variacao_absoluta"]
        / receita_produto["receita_maio"]
    ) * 100,
    np.nan
)

# Ordenar pelos produtos com maiores perdas em reais
receita_produto = receita_produto.sort_values(
    "variacao_absoluta"
)

print("Quantidade de produtos analisados:", len(receita_produto))
print(
    "Produtos com redução de receita:",
    (receita_produto["variacao_absoluta"] < 0).sum()
)

print("\n=== 15 produtos com maiores perdas ===")
display(receita_produto.head(15).round(2))


Quantidade de produtos analisados: 2413
Produtos com redução de receita: 1243

=== 15 produtos com maiores perdas ===


mes_compra,product_id,categoria,receita_maio,receita_junho,variacao_absoluta,variacao_percentual
801,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,"11,820.2000","1,071.0000","-10,749.2000",-90.9400
1623,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,"6,430.8000",179.8000,"-6,251.0000",-97.2000
1655,af751fe756699bddfce4fdd67242fc2a,informatica_acessorios,"5,040.0000",0.0000,"-5,040.0000",-100.0000
2004,d285360f29ac7fd97640bf0baef03de0,relogios_presentes,"5,956.3000",983.6000,"-4,972.7000",-83.4900
100,0abc19e41fdc9d6894b8cae651619253,eletronicos,"4,941.0000",0.0000,"-4,941.0000",-100.0000
837,57c9362442482cd2451f2eabf468e532,relogios_presentes,"4,955.9700",869.0000,"-4,086.9700",-82.4700
1294,89b121bee266dcd25688a1ba72eefb61,informatica_acessorios,"3,995.0000",0.0000,"-3,995.0000",-100.0000
42,0563d4cc419141eab2e5b438046e844e,ferramentas_jardim,"3,930.0000",0.0000,"-3,930.0000",-100.0000
1079,6f3b5b605d91b7439c5e3f5a8dffeea7,relogios_presentes,"4,355.0000",457.9000,"-3,897.1000",-89.4900
609,3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,"7,794.8000","4,047.3000","-3,747.5000",-48.0800


In [20]:

# Considerar somente produtos cuja receita diminuiu
perdas_produtos = receita_produto.loc[
    receita_produto["variacao_absoluta"] < 0
].copy()

# Converter variações negativas em valores positivos de perda
perdas_produtos["perda_absoluta"] = (
    -perdas_produtos["variacao_absoluta"]
)

# Ordenar da maior para a menor perda
perdas_produtos = perdas_produtos.sort_values(
    "perda_absoluta",
    ascending=False
).reset_index(drop=True)

# Calcular a participação acumulada nas perdas brutas
perda_total = perdas_produtos["perda_absoluta"].sum()

perdas_produtos["perda_acumulada"] = (
    perdas_produtos["perda_absoluta"].cumsum()
)

perdas_produtos["participacao_acumulada_pct"] = (
    perdas_produtos["perda_acumulada"] / perda_total
) * 100

# Quantos produtos são necessários para atingir cada patamar?
print(f"Produtos com redução de receita: {len(perdas_produtos)}")
print(f"Perdas brutas somadas: R$ {perda_total:,.2f}")

for percentual in [50, 75, 90]:
    quantidade = (
        perdas_produtos["participacao_acumulada_pct"] < percentual
    ).sum() + 1

    quantidade = min(quantidade, len(perdas_produtos))

    participacao_real = perdas_produtos.loc[
        quantidade - 1, "participacao_acumulada_pct"
    ]

    print(
        f"Para atingir {percentual}% das perdas: "
        f"{quantidade} produtos "
        f"(participação acumulada: {participacao_real:.2f}%)"
    )

print("\n=== 15 produtos com maiores perdas acumuladas ===")
display(
    perdas_produtos[
        [
            "product_id",
            "categoria",
            "receita_maio",
            "receita_junho",
            "perda_absoluta",
            "participacao_acumulada_pct"
        ]
    ].head(15).round(2)
)


Produtos com redução de receita: 1243
Perdas brutas somadas: R$ 336,965.65
Para atingir 50% das perdas: 105 produtos (participação acumulada: 50.13%)
Para atingir 75% das perdas: 326 produtos (participação acumulada: 75.01%)
Para atingir 90% das perdas: 635 produtos (participação acumulada: 90.03%)

=== 15 produtos com maiores perdas acumuladas ===


mes_compra,product_id,categoria,receita_maio,receita_junho,perda_absoluta,participacao_acumulada_pct
0,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,"11,820.2000","1,071.0000","10,749.2000",3.1900
1,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,"6,430.8000",179.8000,"6,251.0000",5.0500
2,af751fe756699bddfce4fdd67242fc2a,informatica_acessorios,"5,040.0000",0.0000,"5,040.0000",6.5400
3,d285360f29ac7fd97640bf0baef03de0,relogios_presentes,"5,956.3000",983.6000,"4,972.7000",8.0200
4,0abc19e41fdc9d6894b8cae651619253,eletronicos,"4,941.0000",0.0000,"4,941.0000",9.4800
5,57c9362442482cd2451f2eabf468e532,relogios_presentes,"4,955.9700",869.0000,"4,086.9700",10.7000
6,89b121bee266dcd25688a1ba72eefb61,informatica_acessorios,"3,995.0000",0.0000,"3,995.0000",11.8800
7,0563d4cc419141eab2e5b438046e844e,ferramentas_jardim,"3,930.0000",0.0000,"3,930.0000",13.0500
8,6f3b5b605d91b7439c5e3f5a8dffeea7,relogios_presentes,"4,355.0000",457.9000,"3,897.1000",14.2000
9,3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,"7,794.8000","4,047.3000","3,747.5000",15.3200


In [21]:

# Selecionar os 105 produtos que concentram aproximadamente 50% das perdas
produtos_prioritarios = perdas_produtos.head(105)[
    ["product_id", "categoria"]
].copy()

itens_prioritarios_105 = itens_com_categoria[
    itens_com_categoria["product_id"].isin(
        produtos_prioritarios["product_id"]
    )
    & itens_com_categoria["mes_compra"].astype(str).isin(
        ["2018-05", "2018-06"]
    )
].copy()

# Agregar receita e quantidade de pedidos por produto e mês
resumo_produto_mes = (
    itens_prioritarios_105
    .groupby(
        [
            "product_id",
            itens_prioritarios_105["mes_compra"].astype(str)
        ]
    )
    .agg(
        receita=("price", "sum"),
        pedidos=("order_id", "nunique")
    )
    .reset_index()
    .rename(columns={"mes_compra": "mes"})
)

# Separar as métricas de maio
maio_produtos = (
    resumo_produto_mes[resumo_produto_mes["mes"] == "2018-05"]
    [["product_id", "receita", "pedidos"]]
    .rename(columns={
        "receita": "receita_maio",
        "pedidos": "pedidos_maio"
    })
)

# Separar as métricas de junho
junho_produtos = (
    resumo_produto_mes[resumo_produto_mes["mes"] == "2018-06"]
    [["product_id", "receita", "pedidos"]]
    .rename(columns={
        "receita": "receita_junho",
        "pedidos": "pedidos_junho"
    })
)

# Garantir que todos os 105 produtos permaneçam na análise
analise_105 = (
    produtos_prioritarios
    .merge(maio_produtos, on="product_id", how="left")
    .merge(junho_produtos, on="product_id", how="left")
)

colunas_numericas = [
    "receita_maio", "receita_junho",
    "pedidos_maio", "pedidos_junho"
]

analise_105[colunas_numericas] = (
    analise_105[colunas_numericas].fillna(0)
)

# Calcular as variações
analise_105["variacao_receita"] = (
    analise_105["receita_junho"]
    - analise_105["receita_maio"]
)

analise_105["variacao_pedidos"] = (
    analise_105["pedidos_junho"]
    - analise_105["pedidos_maio"]
)

analise_105["receita_por_pedido_maio"] = np.where(
    analise_105["pedidos_maio"] > 0,
    analise_105["receita_maio"] / analise_105["pedidos_maio"],
    np.nan
)

analise_105["receita_por_pedido_junho"] = np.where(
    analise_105["pedidos_junho"] > 0,
    analise_105["receita_junho"] / analise_105["pedidos_junho"],
    np.nan
)

# Resumir os padrões observados
print("Total de produtos prioritários:", len(analise_105))

print(
    "Produtos sem pedidos em junho:",
    (
        (analise_105["pedidos_maio"] > 0)
        & (analise_105["pedidos_junho"] == 0)
    ).sum()
)

print(
    "Produtos com menos pedidos em junho:",
    (
        analise_105["pedidos_junho"]
        < analise_105["pedidos_maio"]
    ).sum()
)

print(
    "Produtos com mais pedidos em junho:",
    (
        analise_105["pedidos_junho"]
        > analise_105["pedidos_maio"]
    ).sum()
)

print("\n=== 15 produtos prioritários com maiores perdas ===")
display(
    analise_105.sort_values("variacao_receita")[
        [
            "product_id",
            "categoria",
            "receita_maio",
            "receita_junho",
            "pedidos_maio",
            "pedidos_junho",
            "receita_por_pedido_maio",
            "receita_por_pedido_junho",
            "variacao_receita"
        ]
    ].head(15).round(2)
)


Total de produtos prioritários: 105
Produtos sem pedidos em junho: 76
Produtos com menos pedidos em junho: 104
Produtos com mais pedidos em junho: 1

=== 15 produtos prioritários com maiores perdas ===


,product_id,categoria,receita_maio,receita_junho,pedidos_maio,pedidos_junho,receita_por_pedido_maio,receita_por_pedido_junho,variacao_receita
0,53b36df67ebb7c41585e8d54d6772e08,relogios_presentes,"11,820.2000","1,071.0000",107,9.0000,110.4700,119.0000,"-10,749.2000"
1,aca2eb7d00ea1a7b8ebd4e68314663af,moveis_decoracao,"6,430.8000",179.8000,74,1.0000,86.9000,179.8000,"-6,251.0000"
2,af751fe756699bddfce4fdd67242fc2a,informatica_acessorios,"5,040.0000",0.0000,3,0.0000,"1,680.0000",NaN,"-5,040.0000"
3,d285360f29ac7fd97640bf0baef03de0,relogios_presentes,"5,956.3000",983.6000,23,4.0000,258.9700,245.9000,"-4,972.7000"
4,0abc19e41fdc9d6894b8cae651619253,eletronicos,"4,941.0000",0.0000,2,0.0000,"2,470.5000",NaN,"-4,941.0000"
5,57c9362442482cd2451f2eabf468e532,relogios_presentes,"4,955.9700",869.0000,5,1.0000,991.1900,869.0000,"-4,086.9700"
6,89b121bee266dcd25688a1ba72eefb61,informatica_acessorios,"3,995.0000",0.0000,46,0.0000,86.8500,NaN,"-3,995.0000"
7,0563d4cc419141eab2e5b438046e844e,ferramentas_jardim,"3,930.0000",0.0000,1,0.0000,"3,930.0000",NaN,"-3,930.0000"
8,6f3b5b605d91b7439c5e3f5a8dffeea7,relogios_presentes,"4,355.0000",457.9000,30,3.0000,145.1700,152.6300,"-3,897.1000"
9,3dd2a17168ec895c781a9191c1e95ad7,informatica_acessorios,"7,794.8000","4,047.3000",44,27.0000,177.1500,149.9000,"-3,747.5000"


In [22]:

# Classificar os 105 produtos conforme a presença em pedidos

analise_105["situacao_pedidos"] = np.select(
    [
        analise_105["pedidos_junho"] == 0,
        analise_105["pedidos_junho"] < analise_105["pedidos_maio"],
        analise_105["pedidos_junho"] > analise_105["pedidos_maio"]
    ],
    [
        "Sem pedidos em junho",
        "Menos pedidos em junho",
        "Mais pedidos em junho"
    ],
    default="Mesmo número de pedidos"
)

# Somar as perdas por situação
resumo_situacoes = (
    analise_105
    .groupby("situacao_pedidos")
    .agg(
        quantidade_produtos=("product_id", "nunique"),
        receita_maio=("receita_maio", "sum"),
        receita_junho=("receita_junho", "sum"),
        variacao_receita=("variacao_receita", "sum")
    )
)

resumo_situacoes["perda_absoluta"] = (
    -resumo_situacoes["variacao_receita"].clip(upper=0)
)

perda_total_105 = -analise_105["variacao_receita"].sum()

resumo_situacoes["participacao_perda_pct"] = np.where(
    perda_total_105 > 0,
    resumo_situacoes["perda_absoluta"] / perda_total_105 * 100,
    0
)

print(f"Perda líquida dos 105 produtos: R$ {perda_total_105:,.2f}")

display(resumo_situacoes.round(2))

# Verificar a contribuição dos produtos sem pedidos em junho
sem_pedidos_junho = analise_105[
    analise_105["pedidos_junho"] == 0
]

perda_sem_pedidos = -sem_pedidos_junho["variacao_receita"].sum()

print(
    "\nPerda dos produtos sem pedidos em junho: "
    f"R$ {perda_sem_pedidos:,.2f}"
)

print(
    "Participação na perda dos 105 produtos: "
    f"{perda_sem_pedidos / perda_total_105 * 100:.2f}%"
)


Perda líquida dos 105 produtos: R$ 168,906.62


,quantidade_produtos,receita_maio,receita_junho,variacao_receita,perda_absoluta,participacao_perda_pct
situacao_pedidos,,,,,,
Mais pedidos em junho,1,"1,980.2200",747.7400,"-1,232.4800","1,232.4800",0.7300
Menos pedidos em junho,28,"88,297.0000","24,865.8800","-63,431.1200","63,431.1200",37.5500
Sem pedidos em junho,76,"104,243.0200",0.0000,"-104,243.0200","104,243.0200",61.7200



Perda dos produtos sem pedidos em junho: R$ 104,243.02
Participação na perda dos 105 produtos: 61.72%


In [23]:
print("=== CONFERÊNCIA DOS 105 PRODUTOS ===")

print(
    "Receita de maio: R$ "
    f"{analise_105['receita_maio'].sum():,.2f}"
)

print(
    "Receita de junho: R$ "
    f"{analise_105['receita_junho'].sum():,.2f}"
)

print(
    "Variação total: R$ "
    f"{analise_105['variacao_receita'].sum():,.2f}"
)

print(
    "Perda dos produtos sem pedidos em junho: R$ "
    f"{-analise_105.loc[analise_105['pedidos_junho'] == 0, 'variacao_receita'].sum():,.2f}"
)

=== CONFERÊNCIA DOS 105 PRODUTOS ===
Receita de maio: R$ 194,520.24
Receita de junho: R$ 25,613.62
Variação total: R$ -168,906.62
Perda dos produtos sem pedidos em junho: R$ 104,243.02


In [24]:
# Categorias dos 105 produtos prioritários:
# receita, pedidos e perda de maio para junho

resumo_categoria_105 = (
    analise_105
    .groupby("categoria")
    .agg(
        quantidade_produtos=("product_id", "nunique"),
        receita_maio=("receita_maio", "sum"),
        receita_junho=("receita_junho", "sum"),
        pedidos_maio=("pedidos_maio", "sum"),
        pedidos_junho=("pedidos_junho", "sum"),
        variacao_receita=("variacao_receita", "sum")
    )
    .reset_index()
)

resumo_categoria_105["perda_absoluta"] = (
    -resumo_categoria_105["variacao_receita"].clip(upper=0)
)

resumo_categoria_105["sem_pedidos_junho"] = (
    analise_105["pedidos_junho"].eq(0)
    .groupby(analise_105["categoria"])
    .sum()
    .reindex(resumo_categoria_105["categoria"])
    .to_numpy()
)

resumo_categoria_105 = resumo_categoria_105.sort_values(
    "perda_absoluta",
    ascending=False
)

display(resumo_categoria_105.round(2))

,categoria,quantidade_produtos,receita_maio,receita_junho,pedidos_maio,pedidos_junho,variacao_receita,perda_absoluta,sem_pedidos_junho
9,relogios_presentes,33,"74,007.7000","11,562.7800",328,67.0000,"-62,444.9200","62,444.9200",21
5,informatica_acessorios,10,"25,907.5600","4,047.3000",114,27.0000,"-21,860.2600","21,860.2600",9
4,ferramentas_jardim,13,"21,532.7200",898.2000,69,12.0000,"-20,634.5200","20,634.5200",11
1,cool_stuff,14,"16,006.8000","1,784.5100",85,15.0000,"-14,222.2900","14,222.2900",11
3,esporte_lazer,11,"12,366.8000","1,296.0000",24,3.0000,"-11,070.8000","11,070.8000",10
6,moveis_decoracao,4,"10,070.8000","1,229.5000",100,7.0000,"-8,841.3000","8,841.3000",0
7,moveis_escritorio,6,"9,611.8200","1,287.4400",21,8.0000,"-8,324.3800","8,324.3800",4
0,brinquedos,9,"8,812.5800","1,268.9100",16,2.0000,"-7,543.6700","7,543.6700",7
2,eletronicos,3,"7,370.8600",239.9800,14,2.0000,"-7,130.8800","7,130.8800",2
8,portateis_casa_forno_e_cafe,2,"8,832.6000","1,999.0000",7,1.0000,"-6,833.6000","6,833.6000",1


In [25]:
# Medir a perda dos produtos sem pedidos em junho,
# separadamente para cada categoria

produtos_sem_pedidos_junho = analise_105[
    analise_105["pedidos_junho"] == 0
].copy()

resumo_sem_pedidos_categoria = (
    produtos_sem_pedidos_junho
    .groupby("categoria")
    .agg(
        quantidade_sem_pedidos=("product_id", "nunique"),
        receita_maio=("receita_maio", "sum"),
        receita_junho=("receita_junho", "sum"),
        perda_associada=("variacao_receita", lambda valores: -valores.sum())
    )
    .reset_index()
)

resumo_sem_pedidos_categoria["participacao_perda_105_pct"] = (
    resumo_sem_pedidos_categoria["perda_associada"]
    / perda_total_105
    * 100
)

resumo_sem_pedidos_categoria = (
    resumo_sem_pedidos_categoria
    .sort_values("perda_associada", ascending=False)
)

display(resumo_sem_pedidos_categoria.round(2))

print(
    "\nTotal de perda associado aos produtos sem pedidos em junho: "
    f"R$ {resumo_sem_pedidos_categoria['perda_associada'].sum():,.2f}"
)

,categoria,quantidade_sem_pedidos,receita_maio,receita_junho,perda_associada,participacao_perda_105_pct
8,relogios_presentes,21,"28,821.7900",0.0000,"28,821.7900",17.0600
5,informatica_acessorios,9,"18,112.7600",0.0000,"18,112.7600",10.7200
4,ferramentas_jardim,11,"16,991.2200",0.0000,"16,991.2200",10.0600
1,cool_stuff,11,"10,873.6700",0.0000,"10,873.6700",6.4400
3,esporte_lazer,10,"9,675.8000",0.0000,"9,675.8000",5.7300
0,brinquedos,7,"6,063.8800",0.0000,"6,063.8800",3.5900
2,eletronicos,2,"5,691.0000",0.0000,"5,691.0000",3.3700
6,moveis_escritorio,4,"4,577.3000",0.0000,"4,577.3000",2.7100
7,portateis_casa_forno_e_cafe,1,"3,435.6000",0.0000,"3,435.6000",2.0300



Total de perda associado aos produtos sem pedidos em junho: R$ 104,243.02


In [26]:
# Identificar os produtos sem pedidos em junho
# que apresentaram as maiores perdas de receita

produtos_sem_pedidos = analise_105[
    analise_105["pedidos_junho"] == 0
].copy()

produtos_sem_pedidos = produtos_sem_pedidos.sort_values(
    "receita_maio",
    ascending=False
)

# Calcular a participação acumulada na perda desse grupo
perda_sem_pedidos_total = produtos_sem_pedidos["receita_maio"].sum()

produtos_sem_pedidos["perda_acumulada"] = (
    produtos_sem_pedidos["receita_maio"].cumsum()
)

produtos_sem_pedidos["participacao_acumulada_pct"] = (
    produtos_sem_pedidos["perda_acumulada"]
    / perda_sem_pedidos_total
    * 100
)

# Exibir os 15 produtos de maior impacto
colunas_resultado = [
    "product_id",
    "categoria",
    "receita_maio",
    "pedidos_maio",
    "receita_junho",
    "pedidos_junho",
    "perda_acumulada",
    "participacao_acumulada_pct"
]

display(
    produtos_sem_pedidos[colunas_resultado]
    .head(15)
    .round(2)
)

,product_id,categoria,receita_maio,pedidos_maio,receita_junho,pedidos_junho,perda_acumulada,participacao_acumulada_pct
2,af751fe756699bddfce4fdd67242fc2a,informatica_acessorios,"5,040.0000",3,0.0000,0.0000,"5,040.0000",4.8300
4,0abc19e41fdc9d6894b8cae651619253,eletronicos,"4,941.0000",2,0.0000,0.0000,"9,981.0000",9.5700
6,89b121bee266dcd25688a1ba72eefb61,informatica_acessorios,"3,995.0000",46,0.0000,0.0000,"13,976.0000",13.4100
7,0563d4cc419141eab2e5b438046e844e,ferramentas_jardim,"3,930.0000",1,0.0000,0.0000,"17,906.0000",17.1800
10,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,portateis_casa_forno_e_cafe,"3,435.6000",4,0.0000,0.0000,"21,341.6000",20.4700
12,7a10781637204d8d10485c71a6108a2e,relogios_presentes,"3,383.0000",17,0.0000,0.0000,"24,724.6000",23.7200
13,8b5227e28272123eefd613a10ffd7704,relogios_presentes,"2,979.0000",6,0.0000,0.0000,"27,703.6000",26.5800
14,86999f823d7791c5796d506ae0ed44ea,ferramentas_jardim,"2,689.0000",1,0.0000,0.0000,"30,392.6000",29.1600
15,1231bc5d1cab6d3f64357eba69240f63,ferramentas_jardim,"2,550.0000",1,0.0000,0.0000,"32,942.6000",31.6000
17,993ac19aa975075775c16ba15426f8be,informatica_acessorios,"2,499.7500",1,0.0000,0.0000,"35,442.3500",34.0000


In [27]:
# Investigar a atividade em junho dos vendedores
# que comercializaram os 15 produtos prioritários em maio

top_15_ids = produtos_sem_pedidos.head(15)["product_id"]

# Vendedores que venderam esses produtos em maio
vendedores_prioritarios = (
    itens_com_categoria[
        itens_com_categoria["product_id"].isin(top_15_ids)
        & (itens_com_categoria["mes_compra"].astype(str) == "2018-05")
    ][["seller_id"]]
    .drop_duplicates()
)

# Atividade desses vendedores em toda a operação,
# considerando todos os produtos vendidos em maio e junho
atividade_vendedores = (
    itens_com_categoria[
        itens_com_categoria["seller_id"].isin(
            vendedores_prioritarios["seller_id"]
        )
        & itens_com_categoria["mes_compra"].astype(str).isin(
            ["2018-05", "2018-06"]
        )
    ]
    .groupby([
        "seller_id",
        itens_com_categoria["mes_compra"].astype(str).rename("mes")
    ])
    .agg(
        pedidos=("order_id", "nunique"),
        itens_vendidos=("product_id", "size"),
        receita=("price", "sum")
    )
    .reset_index()
)

# Separar os indicadores de maio e junho
atividade_maio = (
    atividade_vendedores[atividade_vendedores["mes"] == "2018-05"]
    .drop(columns="mes")
    .rename(columns={
        "pedidos": "pedidos_maio",
        "itens_vendidos": "itens_maio",
        "receita": "receita_maio"
    })
)

atividade_junho = (
    atividade_vendedores[atividade_vendedores["mes"] == "2018-06"]
    .drop(columns="mes")
    .rename(columns={
        "pedidos": "pedidos_junho",
        "itens_vendidos": "itens_junho",
        "receita": "receita_junho"
    })
)

comparacao_vendedores = (
    vendedores_prioritarios
    .merge(atividade_maio, on="seller_id", how="left")
    .merge(atividade_junho, on="seller_id", how="left")
    .fillna(0)
)

comparacao_vendedores["situacao_junho"] = np.where(
    comparacao_vendedores["pedidos_junho"] == 0,
    "Sem pedidos em junho",
    "Com pedidos em junho"
)

print("Quantidade de vendedores investigados:",
      comparacao_vendedores["seller_id"].nunique())

print("\nSituação dos vendedores em junho:")
display(
    comparacao_vendedores["situacao_junho"]
    .value_counts()
    .rename_axis("situacao")
    .to_frame("quantidade_vendedores")
)

print("\nDetalhamento por vendedor:")
display(
    comparacao_vendedores
    .sort_values("receita_maio", ascending=False)
    .round(2)
)

Quantidade de vendedores investigados: 12

Situação dos vendedores em junho:


,quantidade_vendedores
situacao,
Com pedidos em junho,10
Sem pedidos em junho,2



Detalhamento por vendedor:


,seller_id,pedidos_maio,itens_maio,receita_maio,pedidos_junho,itens_junho,receita_junho,situacao_junho
1,2eb70248d66e0e3ef83659f71b244378,92,96,"15,866.6900",11.0000,11.0000,"2,148.0000",Com pedidos em junho
4,966cb4760537b1404caedd472cc610a5,18,18,"14,089.6400",20.0000,20.0000,"13,287.9200",Com pedidos em junho
6,4869f7a5dfa277a7dca6462dcf3b52b2,92,94,"14,027.6000",94.0000,97.0000,"15,769.1000",Com pedidos em junho
2,ba90964cff9b9e0e6f32b23b82465f7b,12,12,"13,547.6000",2.0000,2.0000,"3,198.0000",Com pedidos em junho
8,039e6ad9dae79614493083e241147386,2,2,"6,480.0000",0.0000,0.0000,0.0000,Sem pedidos em junho
11,2bf6a2c1e71bbd29a4ad64e6d3c3629f,2,3,"5,500.0000",3.0000,3.0000,"3,993.9000",Com pedidos em junho
3,eeb6de78f79159600292e314a77cbd18,4,4,"5,356.0000",2.0000,2.0000,"1,098.0000",Com pedidos em junho
7,17e34d8224d27a541263c4c64b11a56b,4,9,"5,336.6000",6.0000,6.0000,"1,141.8700",Com pedidos em junho
5,c72de06d72748d1a0dfb2125be43ba63,3,4,"5,040.0000",7.0000,7.0000,"7,278.0000",Com pedidos em junho
10,9803a40e82e45418ab7fb84091af5231,2,8,"3,155.2500",4.0000,4.0000,344.7500,Com pedidos em junho


In [28]:
# Comparar o desempenho geral dos 12 vendedores
# entre maio e junho de 2018

comparacao_vendedores["variacao_receita"] = (
    comparacao_vendedores["receita_junho"]
    - comparacao_vendedores["receita_maio"]
)

comparacao_vendedores["variacao_receita_pct"] = np.where(
    comparacao_vendedores["receita_maio"] > 0,
    comparacao_vendedores["variacao_receita"]
    / comparacao_vendedores["receita_maio"] * 100,
    np.nan
)

resumo_desempenho_vendedores = pd.Series({
    "quantidade_vendedores": comparacao_vendedores["seller_id"].nunique(),
    "receita_total_maio": comparacao_vendedores["receita_maio"].sum(),
    "receita_total_junho": comparacao_vendedores["receita_junho"].sum(),
    "variacao_receita_total": comparacao_vendedores["variacao_receita"].sum(),
    "vendedores_com_queda_receita": (
        comparacao_vendedores["variacao_receita"] < 0
    ).sum(),
    "vendedores_com_aumento_receita": (
        comparacao_vendedores["variacao_receita"] > 0
    ).sum(),
    "vendedores_sem_receita_junho": (
        comparacao_vendedores["receita_junho"] == 0
    ).sum()
})

print("=== DESEMPENHO GERAL DOS VENDEDORES ===")
display(resumo_desempenho_vendedores.to_frame("valor").round(2))

print("\n=== MAIORES QUEDAS DE RECEITA ===")
display(
    comparacao_vendedores
    .sort_values("variacao_receita")
    [[
        "seller_id",
        "receita_maio",
        "receita_junho",
        "variacao_receita",
        "variacao_receita_pct",
        "situacao_junho"
    ]]
    .round(2)
)

=== DESEMPENHO GERAL DOS VENDEDORES ===


,valor
quantidade_vendedores,12.0000
receita_total_maio,"93,167.1800"
receita_total_junho,"48,479.4400"
variacao_receita_total,"-44,687.7400"
vendedores_com_queda_receita,10.0000
vendedores_com_aumento_receita,2.0000
vendedores_sem_receita_junho,2.0000



=== MAIORES QUEDAS DE RECEITA ===


,seller_id,receita_maio,receita_junho,variacao_receita,variacao_receita_pct,situacao_junho
1,2eb70248d66e0e3ef83659f71b244378,"15,866.6900","2,148.0000","-13,718.6900",-86.4600,Com pedidos em junho
2,ba90964cff9b9e0e6f32b23b82465f7b,"13,547.6000","3,198.0000","-10,349.6000",-76.3900,Com pedidos em junho
8,039e6ad9dae79614493083e241147386,"6,480.0000",0.0000,"-6,480.0000",-100.0000,Sem pedidos em junho
3,eeb6de78f79159600292e314a77cbd18,"5,356.0000","1,098.0000","-4,258.0000",-79.5000,Com pedidos em junho
7,17e34d8224d27a541263c4c64b11a56b,"5,336.6000","1,141.8700","-4,194.7300",-78.6000,Com pedidos em junho
10,9803a40e82e45418ab7fb84091af5231,"3,155.2500",344.7500,"-2,810.5000",-89.0700,Com pedidos em junho
9,52ac6f920b7280e4c6d30dd94de6259b,"2,379.8000",0.0000,"-2,379.8000",-100.0000,Sem pedidos em junho
0,8ed8ea9a57c862f2c18c2fe41f601d56,"2,388.0000",219.9000,"-2,168.1000",-90.7900,Com pedidos em junho
11,2bf6a2c1e71bbd29a4ad64e6d3c3629f,"5,500.0000","3,993.9000","-1,506.1000",-27.3800,Com pedidos em junho
4,966cb4760537b1404caedd472cc610a5,"14,089.6400","13,287.9200",-801.7200,-5.6900,Com pedidos em junho


In [29]:
# Separar a receita dos 15 produtos prioritários
# da receita dos demais produtos dos mesmos vendedores

ids_top_15 = set(
    produtos_sem_pedidos.head(15)["product_id"]
)

ids_vendedores = set(
    comparacao_vendedores["seller_id"]
)

base_vendedores = itens_com_categoria[
    itens_com_categoria["seller_id"].isin(ids_vendedores)
    & itens_com_categoria["mes_compra"].astype(str).isin(
        ["2018-05", "2018-06"]
    )
].copy()

base_vendedores["grupo_produto"] = np.where(
    base_vendedores["product_id"].isin(ids_top_15),
    "15 produtos prioritários",
    "Demais produtos dos vendedores"
)

resumo_receita_grupos = (
    base_vendedores
    .groupby(["grupo_produto", base_vendedores["mes_compra"].astype(str)])
    .agg(
        receita=("price", "sum"),
        quantidade_itens=("order_item_id", "count"),
        pedidos=("order_id", "nunique")
    )
    .reset_index()
    .rename(columns={"mes_compra": "mes"})
)

display(
    resumo_receita_grupos
    .sort_values(["grupo_produto", "mes"])
    .round(2)
)

,grupo_produto,mes,receita,quantidade_itens,pedidos
0,15 produtos prioritários,2018-05,"46,864.1300",107,101
1,Demais produtos dos vendedores,2018-05,"46,303.0500",157,145
2,Demais produtos dos vendedores,2018-06,"48,479.4400",153,150


In [30]:
# Garantir que todos os grupos apareçam nos dois meses,
# inclusive quando a receita for zero

ordem_grupos = [
    "15 produtos prioritários",
    "Demais produtos dos vendedores"
]

ordem_meses = ["2018-05", "2018-06"]

tabela_comparacao = (
    resumo_receita_grupos
    .pivot(
        index="grupo_produto",
        columns="mes",
        values="receita"
    )
    .reindex(index=ordem_grupos, columns=ordem_meses)
    .fillna(0)
)

tabela_comparacao.columns = ["receita_maio", "receita_junho"]

tabela_comparacao["variacao_absoluta"] = (
    tabela_comparacao["receita_junho"]
    - tabela_comparacao["receita_maio"]
)

tabela_comparacao["variacao_percentual"] = np.where(
    tabela_comparacao["receita_maio"] > 0,
    tabela_comparacao["variacao_absoluta"]
    / tabela_comparacao["receita_maio"] * 100,
    np.nan
)

display(tabela_comparacao.round(2))

,receita_maio,receita_junho,variacao_absoluta,variacao_percentual
grupo_produto,,,,
15 produtos prioritários,"46,864.1300",0.0000,"-46,864.1300",-100.0000
Demais produtos dos vendedores,"46,303.0500","48,479.4400","2,176.3900",4.7000


In [31]:
# Analisar a receita de cada combinação produto-vendedor
# em maio e junho de 2018

ids_top_15 = set(
    produtos_sem_pedidos.head(15)["product_id"]
)

base_produto_vendedor = itens_com_categoria[
    itens_com_categoria["product_id"].isin(ids_top_15)
    & itens_com_categoria["mes_compra"].astype(str).isin(
        ["2018-05", "2018-06"]
    )
].copy()

receita_produto_vendedor = (
    base_produto_vendedor
    .groupby([
        "product_id",
        "seller_id",
        base_produto_vendedor["mes_compra"].astype(str).rename("mes")
    ])
    .agg(
        receita=("price", "sum"),
        pedidos=("order_id", "nunique")
    )
    .reset_index()
)

tabela_produto_vendedor = (
    receita_produto_vendedor
    .pivot(
        index=["product_id", "seller_id"],
        columns="mes",
        values="receita"
    )
    .reindex(columns=["2018-05", "2018-06"])
    .fillna(0)
    .rename(columns={
        "2018-05": "receita_maio",
        "2018-06": "receita_junho"
    })
    .reset_index()
)

tabela_produto_vendedor["variacao_receita"] = (
    tabela_produto_vendedor["receita_junho"]
    - tabela_produto_vendedor["receita_maio"]
)

tabela_produto_vendedor = tabela_produto_vendedor.sort_values(
    "receita_maio",
    ascending=False
)

display(tabela_produto_vendedor.head(20).round(2))

print(
    "\nCombinações produto-vendedor em maio:",
    (tabela_produto_vendedor["receita_maio"] > 0).sum()
)

print(
    "Combinações com receita em junho:",
    (tabela_produto_vendedor["receita_junho"] > 0).sum()
)

mes,product_id,seller_id,receita_maio,receita_junho,variacao_receita
11,af751fe756699bddfce4fdd67242fc2a,c72de06d72748d1a0dfb2125be43ba63,"5,040.0000",0.0000,"-5,040.0000"
1,0abc19e41fdc9d6894b8cae651619253,17e34d8224d27a541263c4c64b11a56b,"4,941.0000",0.0000,"-4,941.0000"
7,89b121bee266dcd25688a1ba72eefb61,2eb70248d66e0e3ef83659f71b244378,"3,995.0000",0.0000,"-3,995.0000"
0,0563d4cc419141eab2e5b438046e844e,039e6ad9dae79614493083e241147386,"3,930.0000",0.0000,"-3,930.0000"
16,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,ba90964cff9b9e0e6f32b23b82465f7b,"3,435.6000",0.0000,"-3,435.6000"
5,7a10781637204d8d10485c71a6108a2e,4869f7a5dfa277a7dca6462dcf3b52b2,"3,383.0000",0.0000,"-3,383.0000"
6,86999f823d7791c5796d506ae0ed44ea,eeb6de78f79159600292e314a77cbd18,"2,689.0000",0.0000,"-2,689.0000"
2,1231bc5d1cab6d3f64357eba69240f63,039e6ad9dae79614493083e241147386,"2,550.0000",0.0000,"-2,550.0000"
10,993ac19aa975075775c16ba15426f8be,9803a40e82e45418ab7fb84091af5231,"2,499.7500",0.0000,"-2,499.7500"
8,8b5227e28272123eefd613a10ffd7704,2eb70248d66e0e3ef83659f71b244378,"2,460.0000",0.0000,"-2,460.0000"



Combinações produto-vendedor em maio: 17
Combinações com receita em junho: 0


In [32]:
# Consolidar a perda dos 15 produtos por vendedor

resumo_perda_vendedor = (
    tabela_produto_vendedor
    .groupby("seller_id")
    .agg(
        produtos_com_perda=("product_id", "nunique"),
        receita_maio=("receita_maio", "sum"),
        receita_junho=("receita_junho", "sum"),
        variacao_receita=("variacao_receita", "sum")
    )
    .reset_index()
)

resumo_perda_vendedor["perda_absoluta"] = (
    -resumo_perda_vendedor["variacao_receita"].clip(upper=0)
)

perda_total_top15 = resumo_perda_vendedor["perda_absoluta"].sum()

resumo_perda_vendedor["participacao_perda_pct"] = (
    resumo_perda_vendedor["perda_absoluta"]
    / perda_total_top15
    * 100
)

# Acrescentar a situação geral de cada vendedor em junho
resumo_perda_vendedor = resumo_perda_vendedor.merge(
    comparacao_vendedores[
        ["seller_id", "situacao_junho"]
    ],
    on="seller_id",
    how="left"
)

resumo_perda_vendedor = resumo_perda_vendedor.sort_values(
    "perda_absoluta",
    ascending=False
)

display(resumo_perda_vendedor.round(2))

print(f"\nPerda total dos 15 produtos: R$ {perda_total_top15:,.2f}")
print(
    "Vendedores envolvidos:",
    resumo_perda_vendedor["seller_id"].nunique()
)

,seller_id,produtos_com_perda,receita_maio,receita_junho,variacao_receita,perda_absoluta,participacao_perda_pct,situacao_junho
3,2eb70248d66e0e3ef83659f71b244378,3,"6,990.0000",0.0000,"-6,990.0000","6,990.0000",14.9200,Com pedidos em junho
0,039e6ad9dae79614493083e241147386,2,"6,480.0000",0.0000,"-6,480.0000","6,480.0000",13.8300,Sem pedidos em junho
10,c72de06d72748d1a0dfb2125be43ba63,1,"5,040.0000",0.0000,"-5,040.0000","5,040.0000",10.7500,Com pedidos em junho
1,17e34d8224d27a541263c4c64b11a56b,1,"4,941.0000",0.0000,"-4,941.0000","4,941.0000",10.5400,Com pedidos em junho
2,2bf6a2c1e71bbd29a4ad64e6d3c3629f,2,"4,400.0000",0.0000,"-4,400.0000","4,400.0000",9.3900,Com pedidos em junho
9,ba90964cff9b9e0e6f32b23b82465f7b,1,"3,435.6000",0.0000,"-3,435.6000","3,435.6000",7.3300,Com pedidos em junho
4,4869f7a5dfa277a7dca6462dcf3b52b2,1,"3,383.0000",0.0000,"-3,383.0000","3,383.0000",7.2200,Com pedidos em junho
11,eeb6de78f79159600292e314a77cbd18,1,"2,689.0000",0.0000,"-2,689.0000","2,689.0000",5.7400,Com pedidos em junho
8,9803a40e82e45418ab7fb84091af5231,1,"2,499.7500",0.0000,"-2,499.7500","2,499.7500",5.3300,Com pedidos em junho
6,8ed8ea9a57c862f2c18c2fe41f601d56,1,"2,388.0000",0.0000,"-2,388.0000","2,388.0000",5.1000,Com pedidos em junho



Perda total dos 15 produtos: R$ 46,864.13
Vendedores envolvidos: 12


In [33]:
# Resumir a perda dos produtos prioritários
# conforme a situação geral dos vendedores em junho

resumo_status_vendedores = (
    resumo_perda_vendedor
    .groupby("situacao_junho")
    .agg(
        quantidade_vendedores=("seller_id", "nunique"),
        produtos_com_perda=("produtos_com_perda", "sum"),
        perda_total=("perda_absoluta", "sum")
    )
    .reset_index()
)

resumo_status_vendedores["participacao_perda_pct"] = (
    resumo_status_vendedores["perda_total"]
    / resumo_perda_vendedor["perda_absoluta"].sum()
    * 100
)

display(resumo_status_vendedores.round(2))

# Medir a concentração da perda nos dois maiores vendedores
dois_maiores = (
    resumo_perda_vendedor
    .nlargest(2, "perda_absoluta")
)

perda_dois_maiores = dois_maiores["perda_absoluta"].sum()
perda_total = resumo_perda_vendedor["perda_absoluta"].sum()

print(f"Perda dos dois maiores vendedores: R$ {perda_dois_maiores:,.2f}")
print(f"Participação na perda total: {perda_dois_maiores / perda_total * 100:.2f}%")

,situacao_junho,quantidade_vendedores,produtos_com_perda,perda_total,participacao_perda_pct
0,Com pedidos em junho,10,14,"38,004.3300",81.0900
1,Sem pedidos em junho,2,3,"8,859.8000",18.9100


Perda dos dois maiores vendedores: R$ 13,470.00
Participação na perda total: 28.74%


In [34]:
# Quantificar a concentração das vendas de maio por produto
# entre os vendedores que comercializaram os 15 produtos prioritários.

concentracao_produto_vendedor = (
    base_produto_vendedor
    .groupby("product_id")
    .agg(
        vendedores_em_maio=("seller_id", "nunique"),
        receita_maio=("price", "sum"),
        pedidos_maio=("order_id", "nunique")
    )
    .reset_index()
)

concentracao_produto_vendedor = (
    concentracao_produto_vendedor
    .merge(
        produtos[["product_id", "product_category_name"]],
        on="product_id",
        how="left",
        validate="one_to_one"
    )
    .rename(columns={
        "product_category_name": "categoria"
    })
)

concentracao_produto_vendedor["participacao_receita_top15_pct"] = (
    concentracao_produto_vendedor["receita_maio"]
    / concentracao_produto_vendedor["receita_maio"].sum()
    * 100
)

concentracao_produto_vendedor = (
    concentracao_produto_vendedor
    .sort_values("receita_maio", ascending=False)
)

display(
    concentracao_produto_vendedor.round(2)
)

print(
    "\nProdutos com apenas um vendedor em maio:",
    (
        concentracao_produto_vendedor["vendedores_em_maio"] == 1
    ).sum()
)

print(
    "Produtos com mais de um vendedor em maio:",
    (
        concentracao_produto_vendedor["vendedores_em_maio"] > 1
    ).sum()
)

,product_id,vendedores_em_maio,receita_maio,pedidos_maio,categoria,participacao_receita_top15_pct
10,af751fe756699bddfce4fdd67242fc2a,1,"5,040.0000",3,informatica_acessorios,10.7500
1,0abc19e41fdc9d6894b8cae651619253,1,"4,941.0000",2,eletronicos,10.5400
7,89b121bee266dcd25688a1ba72eefb61,1,"3,995.0000",46,informatica_acessorios,8.5200
0,0563d4cc419141eab2e5b438046e844e,1,"3,930.0000",1,ferramentas_jardim,8.3900
14,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,1,"3,435.6000",4,portateis_casa_forno_e_cafe,7.3300
5,7a10781637204d8d10485c71a6108a2e,1,"3,383.0000",17,relogios_presentes,7.2200
8,8b5227e28272123eefd613a10ffd7704,2,"2,979.0000",6,relogios_presentes,6.3600
6,86999f823d7791c5796d506ae0ed44ea,1,"2,689.0000",1,ferramentas_jardim,5.7400
2,1231bc5d1cab6d3f64357eba69240f63,1,"2,550.0000",1,ferramentas_jardim,5.4400
9,993ac19aa975075775c16ba15426f8be,1,"2,499.7500",1,informatica_acessorios,5.3300



Produtos com apenas um vendedor em maio: 13
Produtos com mais de um vendedor em maio: 2


In [35]:
# Investigar os dois produtos comercializados por
# mais de um vendedor em maio.

ids_multiplos_vendedores = set(
    concentracao_produto_vendedor.loc[
        concentracao_produto_vendedor["vendedores_em_maio"] > 1,
        "product_id"
    ]
)

detalhe_multiplos_vendedores = (
    base_produto_vendedor[
        base_produto_vendedor["product_id"].isin(
            ids_multiplos_vendedores
        )
    ]
    .groupby(["product_id", "seller_id"])
    .agg(
        receita_maio=("price", "sum"),
        pedidos_maio=("order_id", "nunique")
    )
    .reset_index()
)

detalhe_multiplos_vendedores = (
    detalhe_multiplos_vendedores
    .merge(
        concentracao_produto_vendedor[
            ["product_id", "categoria"]
        ],
        on="product_id",
        how="left",
        validate="many_to_one"
    )
    .sort_values(
        ["product_id", "receita_maio"],
        ascending=[True, False]
    )
)

display(detalhe_multiplos_vendedores.round(2))

,product_id,seller_id,receita_maio,pedidos_maio,categoria
0,8b5227e28272123eefd613a10ffd7704,2eb70248d66e0e3ef83659f71b244378,"2,460.0000",5,relogios_presentes
1,8b5227e28272123eefd613a10ffd7704,966cb4760537b1404caedd472cc610a5,519.0000,1,relogios_presentes
3,b3cd6ea8ef75cbcb2856861b31eb9bc0,966cb4760537b1404caedd472cc610a5,"1,718.9800",3,relogios_presentes
2,b3cd6ea8ef75cbcb2856861b31eb9bc0,2eb70248d66e0e3ef83659f71b244378,535.0000,1,relogios_presentes


In [38]:
# Consolidar os indicadores dos 15 produtos prioritários

resumo_final_top15 = concentracao_produto_vendedor[
    [
        "product_id",
        "categoria",
        "vendedores_em_maio",
        "receita_maio",
        "pedidos_maio",
        "participacao_receita_top15_pct"
    ]
].copy()

# Incorporar a receita de junho
resumo_final_top15 = resumo_final_top15.merge(
    receita_produto[
        ["product_id", "receita_junho"]
    ],
    on="product_id",
    how="left",
    validate="one_to_one"
)

# Produtos sem receita em junho recebem valor zero
resumo_final_top15["receita_junho"] = (
    resumo_final_top15["receita_junho"].fillna(0)
)

# Calcular a variação e a perda absoluta
resumo_final_top15["variacao_receita"] = (
    resumo_final_top15["receita_junho"]
    - resumo_final_top15["receita_maio"]
)

resumo_final_top15["perda_absoluta"] = (
    -resumo_final_top15["variacao_receita"].clip(upper=0)
)

# Ordenar da maior para a menor perda
resumo_final_top15 = resumo_final_top15.sort_values(
    "perda_absoluta",
    ascending=False
)

# Exibir a tabela consolidada
display(resumo_final_top15.round(2))

print(
    "\nReceita em maio: "
    f"R$ {resumo_final_top15['receita_maio'].sum():,.2f}"
)

print(
    "Receita em junho: "
    f"R$ {resumo_final_top15['receita_junho'].sum():,.2f}"
)

print(
    "Perda líquida: "
    f"R$ {resumo_final_top15['variacao_receita'].sum() * -1:,.2f}"
)

print(
    "Perda absoluta dos produtos: "
    f"R$ {resumo_final_top15['perda_absoluta'].sum():,.2f}"
)

,product_id,categoria,vendedores_em_maio,receita_maio,pedidos_maio,participacao_receita_top15_pct,receita_junho,variacao_receita,perda_absoluta
0,af751fe756699bddfce4fdd67242fc2a,informatica_acessorios,1,"5,040.0000",3,10.7500,0.0000,"-5,040.0000","5,040.0000"
1,0abc19e41fdc9d6894b8cae651619253,eletronicos,1,"4,941.0000",2,10.5400,0.0000,"-4,941.0000","4,941.0000"
2,89b121bee266dcd25688a1ba72eefb61,informatica_acessorios,1,"3,995.0000",46,8.5200,0.0000,"-3,995.0000","3,995.0000"
3,0563d4cc419141eab2e5b438046e844e,ferramentas_jardim,1,"3,930.0000",1,8.3900,0.0000,"-3,930.0000","3,930.0000"
4,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,portateis_casa_forno_e_cafe,1,"3,435.6000",4,7.3300,0.0000,"-3,435.6000","3,435.6000"
5,7a10781637204d8d10485c71a6108a2e,relogios_presentes,1,"3,383.0000",17,7.2200,0.0000,"-3,383.0000","3,383.0000"
6,8b5227e28272123eefd613a10ffd7704,relogios_presentes,2,"2,979.0000",6,6.3600,0.0000,"-2,979.0000","2,979.0000"
7,86999f823d7791c5796d506ae0ed44ea,ferramentas_jardim,1,"2,689.0000",1,5.7400,0.0000,"-2,689.0000","2,689.0000"
8,1231bc5d1cab6d3f64357eba69240f63,ferramentas_jardim,1,"2,550.0000",1,5.4400,0.0000,"-2,550.0000","2,550.0000"
9,993ac19aa975075775c16ba15426f8be,informatica_acessorios,1,"2,499.7500",1,5.3300,0.0000,"-2,499.7500","2,499.7500"



Receita em maio: R$ 46,864.13
Receita em junho: R$ 0.00
Perda líquida: R$ 46,864.13
Perda absoluta dos produtos: R$ 46,864.13


In [39]:
# Comparar a receita dos 15 produtos prioritários com
# a receita dos demais produtos dos mesmos vendedores.

ids_top15 = set(resumo_final_top15["product_id"])

vendedores_top15 = set(
    base_produto_vendedor.loc[
        base_produto_vendedor["product_id"].isin(ids_top15),
        "seller_id"
    ]
)

base_comparacao = itens_com_categoria[
    itens_com_categoria["seller_id"].isin(vendedores_top15)
    & itens_com_categoria["mes_compra"].astype(str).isin(
        ["2018-05", "2018-06"]
    )
].copy()

base_comparacao["grupo_produto"] = np.where(
    base_comparacao["product_id"].isin(ids_top15),
    "15 produtos prioritários",
    "Demais produtos dos mesmos vendedores"
)

comparacao_receita_grupos = (
    base_comparacao
    .groupby(["grupo_produto", base_comparacao["mes_compra"].astype(str).rename("mes")])
    .agg(
        receita=("price", "sum"),
        itens_vendidos=("order_item_id", "count"),
        pedidos=("order_id", "nunique")
    )
    .reset_index()
)

tabela_comparacao = (
    comparacao_receita_grupos
    .pivot(
        index="grupo_produto",
        columns="mes",
        values="receita"
    )
    .reindex(columns=["2018-05", "2018-06"])
    .fillna(0)
    .rename(columns={
        "2018-05": "receita_maio",
        "2018-06": "receita_junho"
    })
    .reset_index()
)

tabela_comparacao["variacao_absoluta"] = (
    tabela_comparacao["receita_junho"]
    - tabela_comparacao["receita_maio"]
)

tabela_comparacao["variacao_percentual"] = np.where(
    tabela_comparacao["receita_maio"] != 0,
    tabela_comparacao["variacao_absoluta"]
    / tabela_comparacao["receita_maio"] * 100,
    np.nan
)

display(tabela_comparacao.round(2))

mes,grupo_produto,receita_maio,receita_junho,variacao_absoluta,variacao_percentual
0,15 produtos prioritários,"46,864.1300",0.0000,"-46,864.1300",-100.0000
1,Demais produtos dos mesmos vendedores,"46,303.0500","48,479.4400","2,176.3900",4.7000


In [40]:
# Validar a quantidade de itens vendidos e pedidos
# dos 15 produtos prioritários em maio e junho.

ids_top15 = set(resumo_final_top15["product_id"])

validacao_top15 = (
    itens_com_categoria[
        itens_com_categoria["product_id"].isin(ids_top15)
        & itens_com_categoria["mes_compra"].astype(str).isin(
            ["2018-05", "2018-06"]
        )
    ]
    .groupby(
        itens_com_categoria["mes_compra"].astype(str).rename("mes")
    )
    .agg(
        itens_vendidos=("order_item_id", "count"),
        pedidos=("order_id", "nunique"),
        produtos_vendidos=("product_id", "nunique"),
        vendedores_ativos=("seller_id", "nunique"),
        receita=("price", "sum")
    )
    .reindex(["2018-05", "2018-06"], fill_value=0)
    .rename_axis("mes")
    .reset_index()
)

display(validacao_top15)

,mes,itens_vendidos,pedidos,produtos_vendidos,vendedores_ativos,receita
0,2018-05,107,101,15,12,"46,864.1300"
1,2018-06,0,0,0,0,0.0000


In [41]:
# Recriar a tabela de validação com os nomes das colunas explícitos

validacao_top15 = (
    itens_com_categoria[
        itens_com_categoria["product_id"].isin(ids_top15)
        & itens_com_categoria["mes_compra"].astype(str).isin(
            ["2018-05", "2018-06"]
        )
    ]
    .assign(
        mes=lambda df: df["mes_compra"].astype(str)
    )
    .groupby("mes")
    .agg(
        itens_vendidos=("order_item_id", "count"),
        pedidos=("order_id", "nunique"),
        produtos_vendidos=("product_id", "nunique"),
        vendedores_ativos=("seller_id", "nunique"),
        receita=("price", "sum")
    )
    .reindex(["2018-05", "2018-06"], fill_value=0)
    .rename_axis("mes")
    .reset_index()
)

display(validacao_top15.round(2))

,mes,itens_vendidos,pedidos,produtos_vendidos,vendedores_ativos,receita
0,2018-05,107,101,15,12,"46,864.1300"
1,2018-06,0,0,0,0,0.0000


In [42]:
# Exibir os indicadores de cada mês em texto simples

for _, linha in validacao_top15.iterrows():
    print(f"\nMês: {linha['mes']}")
    print(f"Itens vendidos: {linha['itens_vendidos']}")
    print(f"Pedidos distintos: {linha['pedidos']}")
    print(f"Produtos vendidos: {linha['produtos_vendidos']}")
    print(f"Vendedores com vendas: {linha['vendedores_ativos']}")
    print(f"Receita: R$ {linha['receita']:,.2f}")


Mês: 2018-05
Itens vendidos: 107
Pedidos distintos: 101
Produtos vendidos: 15
Vendedores com vendas: 12
Receita: R$ 46,864.13

Mês: 2018-06
Itens vendidos: 0
Pedidos distintos: 0
Produtos vendidos: 0
Vendedores com vendas: 0
Receita: R$ 0.00


In [43]:
# Acompanhar a receita mensal dos 15 produtos prioritários
# de maio a dezembro de 2018.

ids_top15 = set(resumo_final_top15["product_id"])

evolucao_mensal_top15 = (
    itens_com_categoria[
        itens_com_categoria["product_id"].isin(ids_top15)
        & itens_com_categoria["mes_compra"].astype(str).between(
            "2018-05", "2018-12"
        )
    ]
    .assign(
        mes=lambda df: df["mes_compra"].astype(str)
    )
    .groupby("mes")
    .agg(
        receita=("price", "sum"),
        itens_vendidos=("order_item_id", "count"),
        pedidos=("order_id", "nunique"),
        produtos_vendidos=("product_id", "nunique"),
        vendedores_com_vendas=("seller_id", "nunique")
    )
    .reindex(
        [f"2018-{mes:02d}" for mes in range(5, 13)],
        fill_value=0
    )
    .rename_axis("mes")
    .reset_index()
)

display(evolucao_mensal_top15.round(2))

,mes,receita,itens_vendidos,pedidos,produtos_vendidos,vendedores_com_vendas
0,2018-05,"46,864.1300",107,101,15,12
1,2018-06,0.0000,0,0,0,0
2,2018-07,"1,916.0000",4,4,1,1
3,2018-08,0.0000,0,0,0,0
4,2018-09,0.0000,0,0,0,0
5,2018-10,0.0000,0,0,0,0
6,2018-11,0.0000,0,0,0,0
7,2018-12,0.0000,0,0,0,0


In [44]:
# Verificar se os 15 produtos prioritários constam no cadastro.

cadastro_top15 = (
    produtos[
        produtos["product_id"].isin(ids_top15)
    ]
    [["product_id", "product_category_name"]]
    .merge(
        resumo_final_top15[
            ["product_id", "receita_maio", "receita_junho"]
        ],
        on="product_id",
        how="left",
        validate="one_to_one"
    )
    .assign(
        consta_no_cadastro=True,
        categoria=lambda df: df["product_category_name"].fillna(
            "categoria_nao_informada"
        )
    )
    .drop(columns="product_category_name")
)

display(cadastro_top15.round(2))

print(
    "Quantidade de produtos prioritários:",
    len(ids_top15)
)
print(
    "Produtos encontrados no cadastro:",
    cadastro_top15["product_id"].nunique()
)

,product_id,receita_maio,receita_junho,consta_no_cadastro,categoria
0,2f465f0f879ab88842042a85bac3a937,"2,379.8000",0.0000,True,moveis_escritorio
1,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,"3,435.6000",0.0000,True,portateis_casa_forno_e_cafe
2,993ac19aa975075775c16ba15426f8be,"2,499.7500",0.0000,True,informatica_acessorios
3,8b5227e28272123eefd613a10ffd7704,"2,979.0000",0.0000,True,relogios_presentes
4,0abc19e41fdc9d6894b8cae651619253,"4,941.0000",0.0000,True,eletronicos
5,7a10781637204d8d10485c71a6108a2e,"3,383.0000",0.0000,True,relogios_presentes
6,1231bc5d1cab6d3f64357eba69240f63,"2,550.0000",0.0000,True,ferramentas_jardim
7,b3cd6ea8ef75cbcb2856861b31eb9bc0,"2,253.9800",0.0000,True,relogios_presentes
8,86999f823d7791c5796d506ae0ed44ea,"2,689.0000",0.0000,True,ferramentas_jardim
9,564a04f027fb97b53ca08464632db860,"2,200.0000",0.0000,True,relogios_presentes


Quantidade de produtos prioritários: 15
Produtos encontrados no cadastro: 15


In [45]:
# Verificar a atividade dos 12 vendedores que venderam
# os 15 produtos prioritários em maio.

ids_vendedores_top15 = set(
    itens_com_categoria.loc[
        itens_com_categoria["product_id"].isin(ids_top15)
        & (itens_com_categoria["mes_compra"].astype(str) == "2018-05"),
        "seller_id"
    ].dropna().unique()
)

atividade_vendedores = (
    itens_com_categoria[
        itens_com_categoria["seller_id"].isin(ids_vendedores_top15)
        & itens_com_categoria["mes_compra"].astype(str).isin(
            ["2018-05", "2018-06"]
        )
    ]
    .assign(
        mes=lambda df: df["mes_compra"].astype(str)
    )
    .groupby(["seller_id", "mes"])
    .agg(
        receita_total=("price", "sum"),
        pedidos=("order_id", "nunique"),
        produtos_vendidos=("product_id", "nunique")
    )
    .unstack("mes", fill_value=0)
)

atividade_vendedores.columns = [
    f"{indicador}_{mes}"
    for indicador, mes in atividade_vendedores.columns
]

for coluna in [
    "receita_total_2018-05",
    "receita_total_2018-06",
    "pedidos_2018-05",
    "pedidos_2018-06",
    "produtos_vendidos_2018-05",
    "produtos_vendidos_2018-06"
]:
    if coluna not in atividade_vendedores.columns:
        atividade_vendedores[coluna] = 0

atividade_vendedores["situacao_junho"] = np.where(
    atividade_vendedores["receita_total_2018-06"] > 0,
    "Continuou vendendo",
    "Sem vendas registradas"
)

atividade_vendedores["variacao_receita"] = (
    atividade_vendedores["receita_total_2018-06"]
    - atividade_vendedores["receita_total_2018-05"]
)

display(
    atividade_vendedores[
        [
            "receita_total_2018-05",
            "receita_total_2018-06",
            "variacao_receita",
            "pedidos_2018-05",
            "pedidos_2018-06",
            "produtos_vendidos_2018-05",
            "produtos_vendidos_2018-06",
            "situacao_junho"
        ]
    ]
    .sort_values("variacao_receita")
    .round(2)
)

print("Vendedores analisados:", len(atividade_vendedores))
print("\nSituação em junho:")
display(atividade_vendedores["situacao_junho"].value_counts())

,receita_total_2018-05,receita_total_2018-06,variacao_receita,pedidos_2018-05,pedidos_2018-06,produtos_vendidos_2018-05,produtos_vendidos_2018-06,situacao_junho
seller_id,,,,,,,,
2eb70248d66e0e3ef83659f71b244378,"15,866.6900","2,148.0000","-13,718.6900",92,11,12,4,Continuou vendendo
ba90964cff9b9e0e6f32b23b82465f7b,"13,547.6000","3,198.0000","-10,349.6000",12,2,6,2,Continuou vendendo
039e6ad9dae79614493083e241147386,"6,480.0000",0.0000,"-6,480.0000",2,0,2,0,Sem vendas registradas
eeb6de78f79159600292e314a77cbd18,"5,356.0000","1,098.0000","-4,258.0000",4,2,4,2,Continuou vendendo
17e34d8224d27a541263c4c64b11a56b,"5,336.6000","1,141.8700","-4,194.7300",4,6,3,5,Continuou vendendo
9803a40e82e45418ab7fb84091af5231,"3,155.2500",344.7500,"-2,810.5000",2,4,6,3,Continuou vendendo
52ac6f920b7280e4c6d30dd94de6259b,"2,379.8000",0.0000,"-2,379.8000",2,0,1,0,Sem vendas registradas
8ed8ea9a57c862f2c18c2fe41f601d56,"2,388.0000",219.9000,"-2,168.1000",12,1,1,1,Continuou vendendo
2bf6a2c1e71bbd29a4ad64e6d3c3629f,"5,500.0000","3,993.9000","-1,506.1000",2,3,3,3,Continuou vendendo


Vendedores analisados: 12

Situação em junho:


situacao_junho
Continuou vendendo        10
Sem vendas registradas     2
Name: count, dtype: int64

In [46]:
# Quantificar a perda dos 15 produtos prioritários por vendedor.
# A comparação considera apenas esses 15 produtos em maio e junho.

vendas_top15_por_vendedor = (
    itens_com_categoria[
        itens_com_categoria["product_id"].isin(ids_top15)
        & itens_com_categoria["mes_compra"].astype(str).isin(
            ["2018-05", "2018-06"]
        )
    ]
    .assign(
        mes=lambda df: df["mes_compra"].astype(str)
    )
    .groupby(["seller_id", "mes"])
    .agg(
        receita=("price", "sum"),
        itens_vendidos=("order_item_id", "count"),
        produtos_vendidos=("product_id", "nunique")
    )
    .unstack("mes", fill_value=0)
)

vendas_top15_por_vendedor.columns = [
    f"{indicador}_{mes}"
    for indicador, mes in vendas_top15_por_vendedor.columns
]

for coluna in [
    "receita_2018-05",
    "receita_2018-06",
    "itens_vendidos_2018-05",
    "itens_vendidos_2018-06",
    "produtos_vendidos_2018-05",
    "produtos_vendidos_2018-06"
]:
    if coluna not in vendas_top15_por_vendedor.columns:
        vendas_top15_por_vendedor[coluna] = 0

vendas_top15_por_vendedor["perda_receita"] = (
    vendas_top15_por_vendedor["receita_2018-05"]
    - vendas_top15_por_vendedor["receita_2018-06"]
)

perda_total_top15 = vendas_top15_por_vendedor["perda_receita"].sum()

vendas_top15_por_vendedor["participacao_perda_pct"] = (
    vendas_top15_por_vendedor["perda_receita"]
    / perda_total_top15
    * 100
)

vendas_top15_por_vendedor = (
    vendas_top15_por_vendedor
    .sort_values("perda_receita", ascending=False)
)

vendas_top15_por_vendedor["participacao_acumulada_pct"] = (
    vendas_top15_por_vendedor["participacao_perda_pct"].cumsum()
)

display(
    vendas_top15_por_vendedor[
        [
            "receita_2018-05",
            "receita_2018-06",
            "perda_receita",
            "participacao_perda_pct",
            "participacao_acumulada_pct",
            "itens_vendidos_2018-05",
            "itens_vendidos_2018-06"
        ]
    ].round(2)
)

print(f"Perda líquida dos 15 produtos: R$ {perda_total_top15:,.2f}")

,receita_2018-05,receita_2018-06,perda_receita,participacao_perda_pct,participacao_acumulada_pct,itens_vendidos_2018-05,itens_vendidos_2018-06
seller_id,,,,,,,
2eb70248d66e0e3ef83659f71b244378,"6,990.0000",0,"6,990.0000",14.9200,14.9200,56,0
039e6ad9dae79614493083e241147386,"6,480.0000",0,"6,480.0000",13.8300,28.7400,2,0
c72de06d72748d1a0dfb2125be43ba63,"5,040.0000",0,"5,040.0000",10.7500,39.5000,4,0
17e34d8224d27a541263c4c64b11a56b,"4,941.0000",0,"4,941.0000",10.5400,50.0400,2,0
2bf6a2c1e71bbd29a4ad64e6d3c3629f,"4,400.0000",0,"4,400.0000",9.3900,59.4300,2,0
ba90964cff9b9e0e6f32b23b82465f7b,"3,435.6000",0,"3,435.6000",7.3300,66.7600,4,0
4869f7a5dfa277a7dca6462dcf3b52b2,"3,383.0000",0,"3,383.0000",7.2200,73.9800,17,0
eeb6de78f79159600292e314a77cbd18,"2,689.0000",0,"2,689.0000",5.7400,79.7200,1,0
9803a40e82e45418ab7fb84091af5231,"2,499.7500",0,"2,499.7500",5.3300,85.0500,1,0


Perda líquida dos 15 produtos: R$ 46,864.13


In [47]:
# Resumir a concentração acumulada das perdas por vendedor.

limiares = [50, 75, 90]

for limiar in limiares:
    quantidade = (
        vendas_top15_por_vendedor[
            vendas_top15_por_vendedor["participacao_acumulada_pct"] < limiar
        ].shape[0] + 1
    )

    quantidade = min(quantidade, len(vendas_top15_por_vendedor))

    participacao_real = vendas_top15_por_vendedor[
        "participacao_perda_pct"
    ].head(quantidade).sum()

    print(
        f"Para alcançar pelo menos {limiar}% da perda, "
        f"são necessários {quantidade} vendedores, "
        f"representando {participacao_real:.2f}% da perda líquida."
    )

Para alcançar pelo menos 50% da perda, são necessários 4 vendedores, representando 50.04% da perda líquida.
Para alcançar pelo menos 75% da perda, são necessários 8 vendedores, representando 79.72% da perda líquida.
Para alcançar pelo menos 90% da perda, são necessários 10 vendedores, representando 90.15% da perda líquida.


In [48]:
# Validação final do diagnóstico dos 15 produtos prioritários.

dados_top15 = itens_com_categoria[
    itens_com_categoria["product_id"].isin(ids_top15)
    & itens_com_categoria["mes_compra"].astype(str).isin(
        ["2018-05", "2018-06"]
    )
].copy()

dados_top15["mes"] = dados_top15["mes_compra"].astype(str)

resumo_validacao = (
    dados_top15
    .groupby("mes")
    .agg(
        receita=("price", "sum"),
        itens_vendidos=("order_item_id", "count"),
        produtos_vendidos=("product_id", "nunique"),
        vendedores_com_vendas=("seller_id", "nunique")
    )
    .reindex(["2018-05", "2018-06"], fill_value=0)
)

perda_total_produtos = (
    resumo_validacao.loc["2018-05", "receita"]
    - resumo_validacao.loc["2018-06", "receita"]
)

perda_total_vendedores = vendas_top15_por_vendedor["perda_receita"].sum()

print("VALIDAÇÃO FINAL — 15 PRODUTOS PRIORITÁRIOS")
print("-" * 48)

for mes in ["2018-05", "2018-06"]:
    linha = resumo_validacao.loc[mes]

    print(f"\nMês: {mes}")
    print(f"Receita: R$ {linha['receita']:,.2f}")
    print(f"Itens vendidos: {linha['itens_vendidos']}")
    print(f"Produtos vendidos: {linha['produtos_vendidos']}")
    print(f"Vendedores com vendas desses produtos: {linha['vendedores_com_vendas']}")

print(f"\nPerda calculada por produto: R$ {perda_total_produtos:,.2f}")
print(f"Perda calculada por vendedor: R$ {perda_total_vendedores:,.2f}")
print(
    "Os valores reconciliam:",
    np.isclose(perda_total_produtos, perda_total_vendedores)
)

VALIDAÇÃO FINAL — 15 PRODUTOS PRIORITÁRIOS
------------------------------------------------

Mês: 2018-05
Receita: R$ 46,864.13
Itens vendidos: 107.0
Produtos vendidos: 15.0
Vendedores com vendas desses produtos: 12.0

Mês: 2018-06
Receita: R$ 0.00
Itens vendidos: 0.0
Produtos vendidos: 0.0
Vendedores com vendas desses produtos: 0.0

Perda calculada por produto: R$ 46,864.13
Perda calculada por vendedor: R$ 46,864.13
Os valores reconciliam: True
